In [5]:
import pandas as pd
import re
import json

# =========================
# 1. LOAD + CLEAN DATA
# =========================
def load_and_clean_csv(file_path):
    df = pd.read_csv(
        file_path,
        sep=';',                 # important for your file
        encoding='latin1',       # fixes weird characters
        engine='python',
        on_bad_lines='skip'
    )

    # Remove empty columns
    df = df.dropna(axis=1, how='all')

    # Strip whitespace
    df.columns = df.columns.str.strip()
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Drop mostly empty rows
    threshold = int(0.7 * len(df.columns))
    df = df.dropna(thresh=threshold)

    print("✅ Loaded & cleaned:", df.shape)
    return df


# =========================
# 2. FIX COLUMN NAMES (AUTO FALLBACK)
# =========================
def simplify_column_name(col):
    col = col.lower()
    col = re.sub(r'[^a-z0-9\s]', '', col)

    stopwords = {
        'the','is','are','am','i','me','my','we','our','you','your',
        'do','does','did','a','an','of','to','in','for','on','with',
        'that','this','it','be','have','has','had','was','were'
    }

    words = [w for w in col.split() if w not in stopwords]
    return "_".join(words[:2]) if words else "col"


def auto_rename(df):
    new_cols = {}
    used = set()

    for col in df.columns:
        new_name = simplify_column_name(col)

        base = new_name
        i = 1
        while new_name in used:
            new_name = f"{base}_{i}"
            i += 1

        used.add(new_name)
        new_cols[col] = new_name

    return df.rename(columns=new_cols), new_cols


# =========================
# 3. MANUAL HIGH-QUALITY MAPPING
# =========================
def apply_custom_mapping(df):
    column_mapping = {
        "timestamp": "timestamp",

        "1_what": "gender",
        "2_age": "age_group",
        "3_number": "population_size",
        "4_municipality": "municipality",
        "5_average": "income_level",
        "6_educational": "education_level",
        "7_experience": "sm_experience",
        "8_frequency": "sm_usage_freq",
        "9_number": "influencers_followed",

        # Convenience
        "10_convenience": "conv_easy",
        "10_convenience_1": "conv_time_saving",
        "10_convenience_2": "conv_accessible",
        "10_convenience_3": "conv_helpful",
        "10_convenience_4": "conv_practical",

        # Interactivity
        "11_interactivity": "interact_engaging",
        "11_interactivity_1": "interact_responsive",
        "11_interactivity_2": "interact_communicative",
        "11_interactivity_3": "interact_active",
        "11_interactivity_4": "interact_dynamic",

        # Attractiveness
        "12_attractiveness": "attr_appealing",
        "12_attractiveness_1": "attr_stylish",
        "12_attractiveness_2": "attr_trendy",
        "12_attractiveness_3": "attr_impressive",

        # Expertise
        "13_expertise": "expert_knowledge",
        "13_expertise_1": "expert_skilled",
        "13_expertise_2": "expert_experienced",
        "13_expertise_3": "expert_credible",

        # Trustworthiness
        "14_trustworthiness": "trust_reliable",
        "14_trustworthiness_1": "trust_honest",
        "14_trustworthiness_2": "trust_dependable",
        "14_trustworthiness_3": "trust_authentic",

        # Attitude (influencers)
        "15_attitude": "infl_likeability",
        "15_attitude_1": "infl_quality",
        "15_attitude_2": "infl_positive",
        "15_attitude_3": "infl_good",

        # Product attitude
        "16_attitudes": "prod_desirable",
        "16_attitudes_1": "prod_pleasant",
        "16_attitudes_2": "prod_likeable",
        "16_attitudes_3": "prod_good",

        # Behavior
        "17_how": "purchase_freq",
        "18_how": "spending_level",

        # Open-ended
        "19_please": "opinion_text",
        "20_who": "favorite_influencers"
    }

    df = df.rename(columns=column_mapping)
    return df, column_mapping


# =========================
# 4. MAIN PIPELINE
# =========================
file_path = "SMI_Attitude_Online_Questionnaire_EN.csv"

# Step 1: Load
df = load_and_clean_csv(file_path)

# Step 2: Auto rename (backup)
df, auto_map = auto_rename(df)

# Step 3: Apply meaningful names
df, custom_map = apply_custom_mapping(df)

# Step 4: Save cleaned dataset
df.to_csv("cleaned_dataset.csv", index=False)

# Step 5: Save mapping for reference
with open("column_mapping.json", "w") as f:
    json.dump(custom_map, f, indent=4)

print("✅ Final dataset saved as cleaned_dataset.csv")
print("✅ Column mapping saved as column_mapping.json")
print("\nFinal Columns:\n", df.columns.tolist())

✅ Loaded & cleaned: (376, 44)
✅ Final dataset saved as cleaned_dataset.csv
✅ Column mapping saved as column_mapping.json

Final Columns:
 ['timestamp', 'gender', 'age_group', 'population_size', 'municipality', 'income_level', 'education_level', 'sm_experience', 'sm_usage_freq', 'influencers_followed', 'conv_easy', 'conv_time_saving', 'conv_accessible', 'conv_helpful', 'conv_practical', 'interact_engaging', 'interact_responsive', 'interact_communicative', 'interact_active', 'interact_dynamic', 'attr_appealing', 'attr_stylish', 'attr_trendy', 'attr_impressive', 'expert_knowledge', 'expert_skilled', 'expert_experienced', 'expert_credible', 'trust_reliable', 'trust_honest', 'trust_dependable', 'trust_authentic', 'infl_likeability', 'infl_quality', 'infl_positive', 'infl_good', 'prod_desirable', 'prod_pleasant', 'prod_likeable', 'prod_good', 'purchase_freq', 'spending_level', 'opinion_text', 'favorite_influencers']


/var/folders/_4/5syx17hn2cg9fynh6n4k9bl40000gn/T/ipykernel_12147/480864663.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [6]:
df

,timestamp,gender,age_group,population_size,municipality,income_level,education_level,sm_experience,sm_usage_freq,influencers_followed,...,infl_positive,infl_good,prod_desirable,prod_pleasant,prod_likeable,prod_good,purchase_freq,spending_level,opinion_text,favorite_influencers
0,1.8.2024 9:01:49,Male,21  30,Less than 1000,Parvomay,Under BGN 1320,Bachelor,More than 5 years,Several times an hour,Between 20 and 30,...,Disagree,Disagree,Agree,Agree,Neither agree nor disagree,Agree,1.0,1.0,NaN,Emil Conrad
1,1.8.2024 9:02:48,Female,21  30,Over 50 000,Asenovgrad,Over BGN 1320,High school,More than 5 years,Several times an hour,Less than 10,...,Disagree,Disagree,Disagree,Disagree,Disagree,Disagree,2.0,2.0,NaN,"I don't have a favorite influencer, I mainly f..."
2,1.8.2024 9:03:32,Male,21  30,Between 1000 and 50 000,Plovdiv,Under BGN 1320,Bachelor,More than 5 years,Several times a week,Between 10 and 20,...,Neither agree nor disagree,Neither agree nor disagree,Strongly disagree,Strongly disagree,Disagree,Neither agree nor disagree,1.0,1.0,NaN,Isabel Ovcharova-instagram
3,1.8.2024 9:04:23,Female,21  30,Between 1000 and 50 000,Karlovo,Under BGN 1320,High school,More than 5 years,Once or twice a day,Less than 10,...,Disagree,Agree,Disagree,Disagree,Agree,Agree,1.0,1.0,NaN,Flora - YouTube and Instagram.\nStefan Popov -...
4,1.8.2024 9:06:27,Female,21  30,Over 50 000,Plovdiv,Over BGN 1320,Bachelor,More than 5 years,Several times a day,Less than 10,...,Agree,Agree,Neither agree nor disagree,Neither agree nor disagree,Agree,Agree,2.0,2.0,NaN,YouTube and TikTok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,1.11.2024 0:50:32,Female,21  30,Between 1000 and 50 000,Batak,Under BGN 1320,High school,Between 3 and 5 years,Several times a day,Between 10 and 20,...,Agree,Agree,Agree,Agree,Agree,Agree,3.0,2.0,NaN,Instagram
372,1.15.2024 23:43:21,Male,Less than 20,Between 1000 and 50 000,Velingrad,Under BGN 1320,Bachelor,More than 5 years,Several times a day,Between 10 and 20,...,Agree,Agree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,3.0,2.0,NaN,I follow them on Instagram
373,1.16.2024 12:05:07,Male,Less than 20,Over 50 000,Plovdiv,Over BGN 1320,High school,More than 5 years,Several times a day,Between 10 and 20,...,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,1.0,4.0,NaN,I have no favorites
374,1.16.2024 12:08:13,Female,21  30,Over 50 000,Stara Zagora,Over BGN 1320,Bachelor,Between 3 and 5 years,Several times a day,Less than 10,...,Strongly agree,Neither agree nor disagree,Agree,Agree,Agree,Neither agree nor disagree,1.0,1.0,Contribute to the faster sale of products due ...,I don't follow influencers
